# 추천시스템 HW1 — 실험 코드


## 1. 작업 폴더 준비
기존 노트북의 폴더와 구분합니다. 출력 로그는 셀 안에 표시됩니다.

In [ ]:
from pathlib import Path
import os, sys, subprocess

ROOT = Path('/content/rs_hw1_code')
ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)
PYTHON = '/content/rs_hw1_code_env/bin/python'

def run(command):
    process = subprocess.Popen(command, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True)
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
        if process.wait() != 0:
            raise RuntimeError('실행 실패: 위 오류를 확인하세요.')
    except BaseException:
        if process.poll() is None:
            process.terminate()
            process.wait()
        raise

print('작업 폴더:', ROOT)


## 2. 라이브러리 설치

In [ ]:
run([sys.executable, '-m', 'pip', 'install', 'uv'])
uv = [sys.executable, '-m', 'uv']
run(uv + ['python', 'install', '3.11'])
if not Path(PYTHON).exists():
    run(uv + ['venv', '--python', '3.11', '/content/rs_hw1_code_env'])
run(uv + ['pip', 'install', '--python', PYTHON, 'torch==2.5.1',
          '--index-url', 'https://download.pytorch.org/whl/cpu'])
packages = ['recbole==1.2.1', 'ray==2.6.3', 'numpy==1.26.4',
            'scipy==1.12.0', 'pandas==2.2.3', 'scikit-learn==1.5.2',
            'matplotlib==3.9.2', 'kmeans-pytorch==0.3', 'setuptools==75.8.0']
run(uv + ['pip', 'install', '--python', PYTHON] + packages)
run(uv + ['pip', 'check', '--python', PYTHON])


## 3. 공통 실험 설정
모든 관측 평점을 implicit 양성 상호작용으로 취급합니다. 사용자별 random 80/10/10 분할과 같은 시드를 사용합니다.
평가는 full-sort이며, NDCG@10을 검증 기준으로 씁니다. `k`는 이웃 수이고 `@10`은 추천 목록 길이입니다.

설정을 바꾸면 아래 `OUTPUT`도 새 폴더 이름으로 바꾸세요. 기존 결과와 새 설정이 섞이는 것을 막습니다.

In [ ]:
%%writefile settings.py
from pathlib import Path

OUTPUT = Path('results')
OUTPUT.mkdir(exist_ok=True)
KS = [1, 5, 10, 100, 500]
COMMON = {
    'seed': 2026, 'reproducibility': True, 'use_gpu': False,
    'worker': 0, 'epochs': 300, 'stopping_step': 10, 'eval_step': 1,
    'train_batch_size': 2048, 'eval_batch_size': 4096,
    'learner': 'adam', 'learning_rate': 0.001, 'weight_decay': 0.0,
    'show_progress': False, 'save_dataset': False, 'save_dataloaders': False,
    'load_col': {'inter': ['user_id', 'item_id', 'rating', 'timestamp']},
    'eval_args': {'split': {'RS': [0.8, 0.1, 0.1]},
                  'group_by': 'user', 'order': 'RO', 'mode': 'full'},
    'metrics': ['NDCG', 'Recall', 'Precision'], 'topk': [10, 20],
    'valid_metric': 'NDCG@10', 'metric_decimal_place': 6,
    'train_neg_sample_args': {'distribution': 'uniform', 'sample_num': 1,
                              'alpha': 1.0, 'dynamic': False, 'candidate_num': 0},
}


## 4. BPR 정규화 구현
RecBole 기본 BPR는 `reg_lambda`를 손실에 사용하지 않으므로, 아래 하위 클래스에서 추가합니다.

배치의 삼중쌍에 대해 평균을 내고, 임베딩 좌표의 제곱은 합합니다. optimizer weight decay는 0입니다.

In [ ]:
%%writefile bpr_l2.py
from recbole.model.general_recommender.bpr import BPR

class BPRL2(BPR):
    def __init__(self, config, dataset):
        super().__init__(config, dataset)
        self.reg_lambda = float(config['reg_lambda'])

    def calculate_loss(self, interaction):
        ranking_loss = super().calculate_loss(interaction)
        user = self.user_embedding(interaction[self.USER_ID])
        positive = self.item_embedding(interaction[self.ITEM_ID])
        negative = self.item_embedding(interaction[self.NEG_ITEM_ID])
        squared_norm = (user.square().sum(dim=1)
                        + positive.square().sum(dim=1)
                        + negative.square().sum(dim=1))
        regularization = squared_norm.mean() / 2
        return ranking_loss + self.reg_lambda * regularization


## 5. 한 설정을 학습·평가하는 공통 함수
모델마다 동일한 분할인지 해시로 확인합니다. 검증 NDCG가 가장 높은 체크포인트로 테스트합니다.
각 실험이 끝날 때 JSON·CSV를 저장하며, 같은 설정을 다시 실행하면 완료 결과를 재사용합니다.
학습 자체를 다시 하려면 호출에 `rerun=True`를 추가하세요.

In [ ]:
%%writefile experiment.py
import hashlib
import json
import time
import importlib.metadata as metadata
import numpy as np
import pandas as pd
import torch
from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.utils import get_model, get_trainer, init_seed
from settings import COMMON, OUTPUT
from bpr_l2 import BPRL2

torch.set_num_threads(4)

def run_one(run_id, model_name, rerun=False, **parameters):
    result_path = OUTPUT / (run_id + '.json')
    signature = {'model': model_name, 'parameters': parameters, 'common': COMMON}
    if result_path.exists() and not rerun:
        previous = json.loads(result_path.read_text())
        if previous['signature'] != signature:
            raise ValueError('설정이 바뀌었습니다. OUTPUT을 새 폴더로 지정하세요.')
        print('완료 결과 재사용:', run_id, flush=True)
        return previous

    config = Config(model=model_name, dataset='ml-100k', config_dict={
        **COMMON, **parameters, 'checkpoint_dir': str(OUTPUT / 'checkpoints' / run_id)})
    init_seed(COMMON['seed'], True)
    dataset = create_dataset(config)
    train, valid, test = data_preparation(config, dataset)
    hashes, counts = [], []
    for loader in (train, valid, test):
        data = loader.dataset.inter_feat
        pairs = np.column_stack([data[config['USER_ID_FIELD']].cpu().numpy(),
                                 data[config['ITEM_ID_FIELD']].cpu().numpy()])
        hashes.append(hashlib.sha256(pairs.astype('<i8').tobytes()).hexdigest())
        counts.append(len(pairs))
    split_path = OUTPUT / 'split.json'
    if split_path.exists() and json.loads(split_path.read_text())['hashes'] != hashes:
        raise ValueError('데이터 분할이 다릅니다. 실험을 중단합니다.')
    split_path.write_text(json.dumps({'hashes': hashes, 'counts': counts}, indent=2))

    init_seed(COMMON['seed'], True)
    cls = BPRL2 if model_name == 'BPR' else get_model(model_name)
    start = time.perf_counter()
    model = cls(config, train.dataset).to(config['device'])
    if model_name == 'ItemKNN':
        assert model.method == parameters['knn_method'], 'KNN 방식이 반영되지 않았습니다.'
    trainer = get_trainer(config['MODEL_TYPE'], config['model'])(config, model)
    _, validation = trainer.fit(train, valid, saved=True, show_progress=False)
    test_scores = trainer.evaluate(test, load_best_model=True, show_progress=False)
    row = {'id': run_id, 'model': model_name, **parameters,
           'seconds': time.perf_counter() - start,
           **{'valid_' + k: float(v) for k, v in validation.items()},
           **{'test_' + k: float(v) for k, v in test_scores.items()}}
    result = {'signature': signature, 'row': row, 'split_hashes': hashes}
    temporary = result_path.with_suffix('.tmp')
    temporary.write_text(json.dumps(result, indent=2))
    temporary.replace(result_path)
    (OUTPUT / (run_id + '_config.txt')).write_text(str(config))
    print(run_id, test_scores, flush=True)
    return result

def save_table(results, name):
    table = pd.DataFrame([result['row'] for result in results])
    table.to_csv(OUTPUT / name, index=False)
    print(table.to_string(index=False), flush=True)
    versions = {p: metadata.version(p) for p in ['recbole','torch','numpy','ray','scipy']}
    (OUTPUT / 'versions.json').write_text(json.dumps(versions, indent=2))


## 6. KNN 실험
ItemKNN과 UserKNN을 각각 k=1, 5, 10, 100, 500으로 실행합니다. UserKNN은 별도 모델명이 아니라 `knn_method="user"` 설정입니다.

In [ ]:
%%writefile run_knn.py
from experiment import run_one, save_table
from settings import KS

results = []
for method in ['item', 'user']:
    for k in KS:
        result = run_one(f'{method}_k{k}', 'ItemKNN', k=k, knn_method=method)
        results.append(result)
        save_table(results, 'knn.csv')


In [ ]:
run([PYTHON, '-u', 'run_knn.py'])


## 7. KNN 그래프 확인
실측 결과를 바로 확인합니다. 숫자 간격이 큰 k는 범주 간격으로 표시합니다.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

knn = pd.read_csv('results/knn.csv')
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, method in zip(axes, ['item', 'user']):
    group = knn[knn['knn_method'] == method].sort_values('k')
    for metric in ['ndcg@10', 'recall@10', 'precision@10']:
        ax.plot(group['k'].astype(str), group['test_' + metric], 'o-', label=metric)
    ax.set(title=method.title() + 'KNN', xlabel='k (categorical spacing)', ylabel='Test score')
    ax.grid(alpha=0.3)
    ax.legend()
fig.tight_layout()
fig.savefig('results/knn.png', dpi=180)
plt.show()


## 8. BPR 실험
① λ=0.1, 임베딩 32/64/128/256 ② 임베딩 64, λ=0.01/0.1/0.5. 중복 설정 (64, 0.1)은 한 번만 실행합니다. 학습 시간이 걸릴 수 있습니다.

In [ ]:
%%writefile run_bpr.py
from experiment import run_one, save_table

settings = [(size, 0.1) for size in [32, 64, 128, 256]]
settings += [(64, 0.01), (64, 0.5)]
results = []
for size, regularization in settings:
    result = run_one(f'bpr_d{size}_r{regularization}', 'BPR',
                     embedding_size=size, reg_lambda=regularization)
    results.append(result)
    save_table(results, 'bpr.csv')


In [ ]:
run([PYTHON, '-u', 'run_bpr.py'])


## 9. BPR 그래프 확인
두 조건에서 NDCG 변화를 확인합니다. 이 그래프는 테스트 수치이며, 최적 설정을 고를 때는 저장된 검증 점수를 사용합니다.

In [ ]:
bpr = pd.read_csv('results/bpr.csv')
by_size = bpr[bpr['reg_lambda'] == 0.1].sort_values('embedding_size')
by_reg = bpr[bpr['embedding_size'] == 64].sort_values('reg_lambda')
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, group, x, title in [
    (axes[0], by_size, 'embedding_size', 'lambda = 0.1'),
    (axes[1], by_reg, 'reg_lambda', 'embedding size = 64')]:
    for metric in ['ndcg@10', 'ndcg@20']:
        ax.plot(group[x].astype(str), group['test_' + metric], 'o-', label=metric)
    ax.set(title=title, xlabel=x + ' (categorical spacing)', ylabel='Test NDCG')
    ax.grid(alpha=0.3)
    ax.legend()
fig.tight_layout()
fig.savefig('results/bpr.png', dpi=180)
plt.show()


## 10. SLIM·EASE 실험
RecBole의 문서 기본 설정을 사용합니다. 이 결과는 특정 설정의 비교이며 모든 모델을 동일한 예산으로 튜닝한 비교는 아닙니다.

In [ ]:
%%writefile run_linear.py
from experiment import run_one, save_table

results = []
results.append(run_one('slim', 'SLIMElastic', alpha=0.2, l1_ratio=0.02,
                       positive_only=True, hide_item=True))
save_table(results, 'linear.csv')
results.append(run_one('ease', 'EASE', reg_weight=250.0))
save_table(results, 'linear.csv')


In [ ]:
run([PYTHON, '-u', 'run_linear.py'])


## 11. ItemKNN·SLIM·EASE 비교표
ItemKNN의 k는 **검증 NDCG@10**으로 고른 뒤, 해당 모델의 테스트 점수를 비교합니다.

In [ ]:
knn = pd.read_csv('results/knn.csv')
linear = pd.read_csv('results/linear.csv')
items = knn[knn['knn_method'] == 'item']
best_item = items.loc[[items['valid_ndcg@10'].idxmax()]]
comparison = pd.concat([best_item, linear], ignore_index=True)
columns = ['id', 'test_ndcg@10', 'test_recall@10', 'test_precision@10']
comparison[columns].to_csv('results/comparison.csv', index=False)
print('검증 점수로 고른 ItemKNN k:', int(best_item.iloc[0]['k']))
display(comparison[columns])


## 12. 결과 다운로드
CSV·그래프·개별 실험 결과·설정·실행 코드를 저장합니다. 모델 체크포인트는 용량 때문에 제외합니다. 이 ZIP을 보관하면 나중에 보고서를 작성할 수 있습니다. 노트북 자체도 Colab 메뉴에서 다운로드하세요.

In [ ]:
from google.colab import files
import zipfile

expected = ['knn.csv', 'bpr.csv', 'linear.csv', 'comparison.csv']
for name in expected:
    assert (ROOT / 'results' / name).exists(), f'결과 누락: {name}'
assert len(pd.read_csv('results/knn.csv')) == 10
assert len(pd.read_csv('results/bpr.csv')) == 6
assert len(pd.read_csv('results/linear.csv')) == 2
archive = ROOT / 'RS_HW1_results.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in sorted(ROOT.rglob('*')):
        if p.is_file() and p != archive and not any(x in p.parts for x in ['checkpoints', '__pycache__']):
            z.write(p, p.relative_to(ROOT))
files.download(str(archive))
